In [15]:
!pip install catboost

In [16]:
!pip install optuna

In [17]:
!pip install autogluon

In [19]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
from autogluon.tabular import TabularPredictor

# ==========================================
# 1. PATHS & DATA
# ==========================================
data_root = "./dataset/public" if os.path.isdir("./dataset/public") else ("./public" if os.path.isdir("./public") else ".")
train_csv = os.path.join(data_root, "train.csv")
test_csv = os.path.join(data_root, "test.csv")
output_dir = "./working"
os.makedirs(output_dir, exist_ok=True)

train = pd.read_csv(train_csv)
test = pd.read_csv(test_csv)
target_col = 'underperforming'
id_col = 'id'

# ==========================================
# 2. FEATURE ENGINEERING HẠNG NẶNG
# ==========================================
print("Đang chế tạo đặc trưng: Phân cụm địa lý và Thống kê phân vị...")
train['is_train'] = 1
test['is_train'] = 0
test[target_col] = -1
df_all = pd.concat([train, test], axis=0, ignore_index=True)

# 2.1. Phân cụm Địa lý (K-Means)
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)
df_all['geo_cluster'] = kmeans.fit_predict(df_all[['latitude', 'longitude']])
df_all['geo_cluster'] = df_all['geo_cluster'].astype('category')

# 2.2. Đặc trưng Nhóm
df_all['cap_percentile'] = df_all.groupby('primary_fuel')['capacity_mw'].rank(pct=True)
df_all['age_percentile'] = df_all.groupby('primary_fuel')['plant_age'].rank(pct=True)
df_all['cap_vs_geo_mean'] = df_all['capacity_mw'] / (df_all.groupby('geo_cluster')['capacity_mw'].transform('mean') + 1e-5)

train_fe = df_all[df_all['is_train'] == 1].drop(columns=['is_train'])
test_fe = df_all[df_all['is_train'] == 0].drop(columns=['is_train', target_col])

# ==========================================
# 3. AUTOGLUON: HUẤN LUYỆN MULTI-LAYER STACKING
# ==========================================
print("\nKích hoạt AutoGluon: Chế độ 'Best Quality' (Có thể mất 5-15 phút)...")

train_data = train_fe.drop(columns=[id_col])
test_data = test_fe.drop(columns=[id_col])

# Đã xóa random_state=42 ở đây để không bị lỗi với bản 1.5.0
predictor = TabularPredictor(
    label=target_col,
    eval_metric='roc_auc',
    verbosity=2
).fit(
    train_data=train_data,
    presets='best_quality',
    time_limit=900
)

print("\n--- BẢNG XẾP HẠNG CÁC MÔ HÌNH BÊN TRONG AUTOGLUON ---")
results = predictor.leaderboard()
print(results)

# ==========================================
# 4. XUẤT FILE NỘP BÀI
# ==========================================
print("\nĐang dự đoán tập test...")
test_preds = predictor.predict_proba(test_data)[1]

submission = pd.DataFrame({
    'id': test_fe[id_col],
    'underperforming': test_preds
})
out_path = os.path.join(output_dir, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Đã lưu file nộp bài tối thượng tại: {out_path}")

No path specified. Models will be saved in: "AutogluonModels/ag-20260227_051129"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.17 GB / 12.67 GB (88.1%)
Disk Space Avail:   74.79 GB / 107.72 GB (69.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable sta

Đang chế tạo đặc trưng: Phân cụm địa lý và Thống kê phân vị...

Kích hoạt AutoGluon: Chế độ 'Best Quality' (Có thể mất 5-15 phút)...


	Running DyStack sub-fit in a ray process to avoid memory leakage. Enabling ray logging (enable_ray_logging=True). Specify `ds_args={'enable_ray_logging': False}` if you experience logging issues.
2026-02-27 05:11:43,523	INFO worker.py:2014 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
		Context path: "/content/AutogluonModels/ag-20260227_051129/ds_sub_fit/sub_fit_ho"
(_dystack pid=14947) Running DyStack sub-fit ...
(_dystack pid=14947) Beginning AutoGluon training ... Time limit = 209s
(_dystack pid=14947) AutoGluon will save models to "/content/AutogluonModels/ag-20260227_051129/ds_sub_fit/sub_fit_ho"
(_dystack pid=14947) Train Data Rows:    5755
(_dystack pid=14947) Train Data Columns: 18
(_dystack pid=14947) Label Column:       underperforming
(_dystack pid=14947) Problem Type:       binary
(_dystack pid=14947) Preprocessing data ...
(_dystack pid=14947) Selected class <--> label mapping:  class 1 = 1, class 0 = 0
(_dystack pid=14947) Using Feature G

(autoscaler +46s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +46s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_dystack pid=14947) 	0.745	 = Validation score   (roc_auc)
(_dystack pid=14947) 	34.96s	 = Training   runtime
(_dystack pid=14947) 	0.51s	 = Validation runtime
(_dystack pid=14947) Fitting model: LightGBM_BAG_L1 ... Training model for up to 96.61s of the 165.96s of remaining time.
(_dystack pid=14947) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.25%)


...

(autoscaler +3m7s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_dystack pid=14947) 	0.7517	 = Validation score   (roc_auc)
(_dystack pid=14947) 	37.94s	 = Training   runtime
(_dystack pid=14947) 	0.34s	 = Validation runtime
(_dystack pid=14947) Fitting model: LightGBM_BAG_L2 ... Training model for up to 14.65s of the 14.56s of remaining time.
(_ray_fit pid=16615) 	Ran out of time, early stopping on iteration 229.
(_dystack pid=14947) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.27%)


(autoscaler +3m42s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_dystack pid=14947) 	0.7508	 = Validation score   (roc_auc)
(_dystack pid=14947) 	32.09s	 = Training   runtime
(_dystack pid=14947) 	0.11s	 = Validation runtime
(_dystack pid=14947) Fitting model: WeightedEnsemble_L3 ... Training model for up to 208.01s of the -26.82s of remaining time.
(_dystack pid=14947) 	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/9.5 GB
(_dystack pid=14947) 	Ensemble Weights: {'RandomForestGini_BAG_L1': 0.28, 'LightGBMXT_BAG_L1': 0.24, 'CatBoost_BAG_L1': 0.24, 'LightGBM_BAG_L2': 0.16, 'RandomForestEntr_BAG_L1': 0.04, 'LightGBMXT_BAG_L2': 0.04}
(_dystack pid=14947) 	0.759	 = Validation score   (roc_auc)
(_dystack pid=14947) 	0.23s	 = Training   runtime
(_dystack pid=14947) 	0.0s	 = Validation runtime
(_dystack pid=14947) AutoGluon training complete, total runtime = 235.71s ... Best model: WeightedEnsemble_L3 | Estimated inference throughput: 450.5 rows/s (720 batch size)
(_dystack pid=14947) TabularPredictor saved. To load, use: predictor = 


--- BẢNG XẾP HẠNG CÁC MÔ HÌNH BÊN TRONG AUTOGLUON ---
                      model  score_val eval_metric  pred_time_val    fit_time  \
0       WeightedEnsemble_L2   0.760091     roc_auc       3.360236  564.060888   
1           CatBoost_BAG_L1   0.742986     roc_auc       0.207874  186.080429   
2   RandomForestGini_BAG_L1   0.740799     roc_auc       0.859985    5.495792   
3         LightGBMXT_BAG_L1   0.740393     roc_auc       0.397223   35.680887   
4   RandomForestEntr_BAG_L1   0.739462     roc_auc       0.397541    7.182142   
5           LightGBM_BAG_L1   0.736014     roc_auc       0.314275   34.004825   
6     ExtraTreesEntr_BAG_L1   0.735492     roc_auc       0.440833    2.599501   
7     ExtraTreesGini_BAG_L1   0.734153     roc_auc       0.458704    2.842870   
8            XGBoost_BAG_L1   0.731901     roc_auc       0.275867   49.073179   
9    NeuralNetFastAI_BAG_L1   0.713736     roc_auc       0.495466  123.444722   
10    NeuralNetTorch_BAG_L1   0.708924     roc_auc    